In [1]:
import os
import os.path
import pickle
import pandas as pd
import numpy as np
from tqdm import tqdm

In [2]:
import os
import sys
from pathlib import Path

# === CONFIGURATION ===
# Choose which dataset to run on: "val" or "test"
DATASET_MODE = "test"  # Change to "test" for final submission

# Set to True to rebuild indices from CSV (required on first run)
# Set to False to load cached indices (faster for subsequent runs)
FORCE_REBUILD_INDICES = False

# Detect environment
KAGGLE_ENV = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if KAGGLE_ENV:
    # Kaggle paths
    DATA_PATH = Path("/kaggle/input/omnilex-data")
    MODEL_PATH = Path("/kaggle/input/llama-model")
    OUTPUT_PATH = Path("/kaggle/working")
    INDEX_PATH = Path("/kaggle/input/omnilex-indices")
    sys.path.insert(0, "/kaggle/input/omnilex-utils")
else:
    # Local development paths
    REPO_ROOT = Path(".").resolve().parent
    DATA_PATH = REPO_ROOT / "data"
    MODEL_PATH = REPO_ROOT / "models"
    OUTPUT_PATH = REPO_ROOT / "output"
    INDEX_PATH = REPO_ROOT / "data" / "processed"

# CSV corpus files for index building
LAWS_CSV = DATA_PATH / "laws_de.csv"
COURTS_CSV = DATA_PATH / "court_considerations.csv"

# Index cache paths
LAWS_INDEX_PATH = INDEX_PATH / "laws_index.pkl"
COURTS_INDEX_PATH = INDEX_PATH / "courts_index.pkl"

# Derived paths based on DATASET_MODE
QUERY_FILE = DATA_PATH / f"{DATASET_MODE}.csv"
IS_VALIDATION_MODE = DATASET_MODE == "val"

# Create output directory
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
INDEX_PATH.mkdir(parents=True, exist_ok=True)

print(f"Environment: {'Kaggle' if KAGGLE_ENV else 'Local'}")
print(f"Dataset mode: {DATASET_MODE}")
print(f"Query file: {QUERY_FILE}")
print(f"Validation mode: {IS_VALIDATION_MODE}")
print(f"Force rebuild indices: {FORCE_REBUILD_INDICES}")
print(f"\nCorpus files:")
print(f"  Laws CSV: {LAWS_CSV} ({LAWS_CSV.stat().st_size / 1e6:.1f} MB)" if LAWS_CSV.exists() else f"  Laws CSV: {LAWS_CSV} (NOT FOUND)")
print(f"  Courts CSV: {COURTS_CSV} ({COURTS_CSV.stat().st_size / 1e9:.2f} GB)" if COURTS_CSV.exists() else f"  Courts CSV: {COURTS_CSV} (NOT FOUND)")
print(f"\nIndex cache: {INDEX_PATH}")

Environment: Local
Dataset mode: test
Query file: /root/autodl-tmp/llm-legal/Omnilex-Agentic-Retrieval-Competition/data/test.csv
Validation mode: False
Force rebuild indices: False

Corpus files:
  Laws CSV: /root/autodl-tmp/llm-legal/Omnilex-Agentic-Retrieval-Competition/data/laws_de.csv (73.0 MB)
  Courts CSV: /root/autodl-tmp/llm-legal/Omnilex-Agentic-Retrieval-Competition/data/court_considerations.csv (2.43 GB)

Index cache: /root/autodl-tmp/llm-legal/Omnilex-Agentic-Retrieval-Competition/data/processed


# 2. Load Corpora and Build/Load Indices

In [3]:
from bm25index import BM25Index
import bm25index

In [4]:
# Load or build laws index
# Laws CSV: ~45MB, ~269K rows
# Build time: ~30 seconds | Load from cache: <1 second

laws_index = bm25index.get_or_build_index(
    name="laws",
    csv_path=LAWS_CSV,
    index_path=LAWS_INDEX_PATH,
    force_rebuild=FORCE_REBUILD_INDICES,
    max_rows=100000000  # Uncomment to test with smaller corpus
)
print(f"\nLaws index: {len(laws_index.documents):,} documents")

# Test search
test_results = laws_index.search("Vertrag", top_k=3)
print(f"\nTest search 'Vertrag': {len(test_results)} results")
if test_results:
    print(test_results)

Loading cached laws index from /root/autodl-tmp/llm-legal/Omnilex-Agentic-Retrieval-Competition/data/processed/laws_index.pkl
  Loaded 175,933 documents

Laws index: 175,933 documents


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

Stem Tokens:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]


Test search 'Vertrag': 1 results
[{'query': 'vertrag', 'hits': [{'rank': 1, 'score': 2.8019, 'text': '4 wird ein rahmen vertrag mit nur einer anbieterin abgeschlossen, so wer den die auf die sem rahmen vertrag beruhenden einzel verträge entsprechend den bedingungen des rahmen vertrags abgeschlossen. für den abschluss der einzel verträge kann die auftraggeberin die jeweilige vertrags partnerin schriftlich auffordern, ihr ange bot zu vervollständigen.', 'citation': 'Art. 25 Abs. 4 BöB'}, {'rank': 2, 'score': 2.783, 'text': '6 ist das bakom registerbetreiberin, so unter steht der vertrag dem öffentlichen recht (verwaltungsrechtlicher vertrag); ist die auf gabe an einen dritten übertragen, so unter steht der vertrag dem privat recht (privatrechtlicher vertrag).', 'citation': 'Art. 17 Abs. 6 VID'}, {'rank': 3, 'score': 2.7276, 'text': '1 durch vertrag kann die verpflichtung zum abschluss eines künftigen vertrages begründet werden.', 'citation': 'Art. 22 Abs. 1 OR'}]}]


In [5]:
# Load or build courts index
# Courts CSV: ~2.3GB, ~2.5M rows
# Full corpus build time: ~15-20 minutes | Load from cache: ~10 seconds
# Full corpus can have peak memory during build: ~8-16GB

courts_index = bm25index.get_or_build_index(
    name="courts",
    csv_path=COURTS_CSV,
    index_path=COURTS_INDEX_PATH,
    force_rebuild=FORCE_REBUILD_INDICES,
    max_rows=100000000  # Change to use bigger corpus
)
print(f"\nCourts index: {len(courts_index.documents):,} documents")

# Test search
test_results = courts_index.search("Meinungsfreiheit", top_k=3)
print(f"\nTest search 'Meinungsfreiheit': {len(test_results)} results")
if test_results:
    print(f"  Top result: {test_results[0].get('citation', 'N/A')}")

Loading cached courts index from /root/autodl-tmp/llm-legal/Omnilex-Agentic-Retrieval-Competition/data/processed/courts_index.pkl
  Loaded 2,476,315 documents

Courts index: 2,476,315 documents


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

Stem Tokens:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]


Test search 'Meinungsfreiheit': 1 results
  Top result: N/A


In [6]:
import re

In [7]:
from FlagEmbedding import FlagReranker, BGEM3FlagModel

dense_model = BGEM3FlagModel('/root/.cache/modelscope/hub/models/BAAI/bge-m3', use_fp16=True)
reranker = FlagReranker('/root/.cache/modelscope/hub/models/BAAI/bge-reranker-v2-m3', use_fp16=True, normalize=True) # Setting use_fp16 to True speeds up computation with a slight performance degradation

In [8]:
import query_by_dense
import citation_utils

court_consideration_df = pd.read_csv("../data/court_considerations.csv")
court_consideration_d = {}
for citation, text in zip(court_consideration_df['citation'].tolist(), court_consideration_df['text'].tolist()):
    # if citation in court_consideration_d:
    #     court_consideration_d[citation] = court_consideration_d[citation] + '\n\n' + text
    # else:
    #     court_consideration_d[citation] = text
    court_consideration_d[citation] = text

law_df = pd.read_csv("../data/laws_de.csv")
law_d = dict(zip(law_df['citation'].tolist(), law_df['text'].tolist()))

test_df = pd.read_csv('../data/test_rewrite_001.csv')


court_doc = [{'citation':citation, 'text':text} for citation,text in zip(court_consideration_df['citation'].tolist(), court_consideration_df['text'].tolist())]

print("data loaded")

data loaded


In [ ]:
import dense_index
from dense_index import DenseIndex

print(dense_model.normalize_embeddings)
court_dense_index = DenseIndex(dense_model, "../data/processed/_dense_court", court_doc)
court_dense_index.info()

True


In [ ]:
from sparse_index import SparseIndex

court_sparse_index = SparseIndex(dense_model, "../data/processed/_sparse_court", court_doc)
court_sparse_index.load()



In [ ]:
import citation_utils
import rerank_utils

id_l = []
citation_l = []

for id, q, q_en in zip(test_df['query_id'].tolist(), test_df['query'].tolist(), test_df['query_en'].tolist()):
    print("query len:", len(q))
    id_l.append(id)
    citations = []

    first_layer_citation = []
    for citation in citation_utils.extract_citations_from_text(q_en):
        first_layer_citation.append(citation)


    # test_results = courts_index.search(q, top_k=1000)[0]['hits']
    test_results = []
    if len(first_layer_citation) > 0:
        print("====> sparse search")
        test_results_sparse = court_sparse_index.search(q, 1000)
        test_results.extend(test_results_sparse)
    else:
        print("====> dense search")
        test_results_dense = court_dense_index.search(q, 1000)
        test_results.extend(test_results_dense)

    _set = set([hit['citation'] for hit in test_results])
    first_layer_citation.extend(list(_set))

    print("first_layer_citation.len:", len(first_layer_citation))

    raw_hits = citation_utils.BFS_citation(court_consideration_d, law_d, first_layer_citation, max_level=2) # 广度优先搜索

    print("raw_hits.len:", len(raw_hits))
    
    court_hits = [hits for hits in raw_hits if hits['citation'] in court_consideration_d]
    law_hits = [hits for hits in raw_hits if hits['citation'] in law_d]

    # court_l = rerank_utils.rerank_by_dense_batch(reranker, q, court_hits, 20, 20)
    court_l = rerank_utils.rerank_by_dense_batch_chunked(reranker, q, court_hits, 30, 20, 384, 128)
    # court_l = query_by_dense.query_by_dense(model, q, court_hits, 20)
    for court, score in court_l:
        citations.append(court['citation'])

    raw_hits_2 = citation_utils.BFS_citation(court_consideration_d, law_d, citations, max_level=2)
    for hits in raw_hits_2:
        citations.append(hits['citation'])

    # # law_l = rerank_utils.rerank_by_dense_batch(reranker, q, law_hits, 20, 20)
    # law_l = rerank_utils.rerank_by_dense_batch_chunked(reranker, q, law_hits, 20, 20, 384, 128)
    # # law_l = query_by_dense.query_by_dense(model, q, law_hits, 20)
    # for law, score in law_l:
    #     if score > 0.2:
    #         citations.append(law['citation'])

    # if len(citations) == 0:
    #     citations.append(court_l[0][0]['citation'])
    #     citations.append(law_l[0][0]['citation'])

    citations = list(set(citations))
    citation_l.append(';'.join(citations))
    print(id, len(citations))

result_df = pd.DataFrame({'query_id':id_l, 'predicted_citations':citation_l})
result_df.to_csv("../data/result.csv", index=False)